# 03. Агент LangChain с MCP-инструментом

Этот ноутбук показывает агента, который использует MCP-инструмент
поиска по базе знаний D&D.

Цепочка:
вопрос пользователя → агент LangChain (GigaChat) →
решает вызвать qdrant-find → MCP-сервер → top-k чанков → ответ

Предварительно должен быть запущен MCP-сервер:
mcp-server-qdrant --transport streamable-http

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path(".env"))

GIGACHAT_CREDENTIALS = os.getenv("GIGACHAT_CREDENTIALS")
GIGACHAT_SCOPE       = os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_PERS")
GIGACHAT_MODEL       = os.getenv("GIGACHAT_MODEL", "GigaChat-2-Max")

print("GIGACHAT_CREDENTIALS:", "ОК: задан" if GIGACHAT_CREDENTIALS else "НЕ ОК: не найден")
print("GIGACHAT_MODEL:", GIGACHAT_MODEL)

GIGACHAT_CREDENTIALS: ОК: задан
GIGACHAT_MODEL: GigaChat-2-Max


## Подключаем MCP-клиент и создаём агента

Агент получает MCP-инструмент qdrant-find как обычный LangChain tool.
GigaChat сам решает когда и с каким запросом его вызвать.

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_gigachat import GigaChat
from langchain.agents import create_agent

# Подключаемся к уже запущенному MCP-серверу
mcp_config = {
    "qdrant-dnd": {
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8000/mcp",
    }
}

client = MultiServerMCPClient(mcp_config)
tools = await client.get_tools()

print(f"MCP-инструментов получено: {len(tools)}")
for t in tools:
    print(f"  - {t.name}: {t.description[:80]}")

MCP-инструментов получено: 2
  - qdrant-find: Look up memories in Qdrant. Use this tool when you need to: 
 - Find memories by
  - qdrant-store: Keep the memory for later use, when you are asked to remember something.


In [6]:
from langchain_gigachat import GigaChat
from langchain.agents import create_agent

# Берём только инструмент поиска — qdrant-store не совместим с GigaChat
search_tools = [t for t in tools if t.name == "qdrant-find"]
print(f"Инструментов для агента: {[t.name for t in search_tools]}")

llm = GigaChat(
    credentials=GIGACHAT_CREDENTIALS,
    scope=GIGACHAT_SCOPE,
    model=GIGACHAT_MODEL,
    verify_ssl_certs=False,
    temperature=0,
)

agent = create_agent(
    model=llm,
    tools=search_tools,
    system_prompt=(
        "Ты эксперт по правилам D&D 5-й редакции. "
        "У тебя есть инструмент поиска по Книге игрока (PHB). "
        "Когда тебя спрашивают о правилах, заклинаниях, классах или механиках — "
        "обязательно используй инструмент поиска чтобы найти точную информацию. "
        "После поиска отвечай на основе найденных фрагментов."
    ),
)

print("Агент создан")
print(f"   Модель: {GIGACHAT_MODEL}")

Инструментов для агента: ['qdrant-find']
Агент создан
   Модель: GigaChat-2-Max


## Запускаем агента с вопросами по D&D

Агент получает вопрос, решает вызвать qdrant-find,
получает фрагменты из PHB и формирует ответ.

In [7]:
async def ask_agent(question: str):
    """Задать вопрос агенту и показать ход рассуждений."""
    print(f"Вопрос: '{question}'")
    print("=" * 60)
    
    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": question}]
    })
    
    # Показываем все сообщения в цепочке
    for msg in result["messages"]:
        if msg.type == "human":
            continue
        elif msg.type == "ai" and hasattr(msg, "tool_calls") and msg.tool_calls:
            print(f"[Агент вызывает инструмент]: {msg.tool_calls[0]['name']}")
            print(f"  Запрос: {msg.tool_calls[0]['args']}")
        elif msg.type == "tool":
            print(f"[MCP вернул]: {str(msg.content)[:200]}...")
        elif msg.type == "ai":
            print(f"\n[Ответ агента]:\n{msg.content}")
    
    print()
    return result

# Первый вопрос
await ask_agent("Какой урон наносит заклинание Огненный шар?")

Вопрос: 'Какой урон наносит заклинание Огненный шар?'
[Агент вызывает инструмент]: qdrant-find
  Запрос: {'query': 'fireball damage'}
[MCP вернул]: [{'type': 'text', 'text': '[\n  "Results for the query \'fireball damage\'",\n  "<entry><content>верки характеристик, сделанных для оценки или \\nисследования мелких и высокодетализированных \\nпредме...

[Ответ агента]:
Заклинание "Огненный шар" наносит урон равный **8к6** единиц огня. Это значение удваивается, если используется ячейка более высокого уровня: добавляется +1к6 за каждую ячейку выше третьей.

Таким образом:
- Базовый урон: $8 \times d6$
- На уровне 4+: $(8+n) \times d6$, где n – количество уровней выше третьего.



{'messages': [HumanMessage(content='Какой урон наносит заклинание Огненный шар?', additional_kwargs={}, response_metadata={}, id='43643722-ff5b-4236-a145-fb29db769b5d'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'qdrant-find', 'arguments': {'query': 'fireball damage'}}, 'functions_state_id': '019ea3c6-ca23-7989-9145-27cf8ea2be6a'}, response_metadata={'token_usage': {'prompt_tokens': 175, 'completion_tokens': 28, 'total_tokens': 203, 'precached_prompt_tokens': 2}, 'model_name': 'GigaChat-2-Max:2.0.28.2', 'x_headers': {'x-request-id': '4bcb49e1-5d73-4269-ba48-4992a18a294e', 'x-session-id': 'be09332f-1ed9-493b-b8c1-6ca403c5e951', 'x-client-id': None}, 'finish_reason': 'function_call'}, id='4bcb49e1-5d73-4269-ba48-4992a18a294e', tool_calls=[{'name': 'qdrant-find', 'args': {'query': 'fireball damage'}, 'id': '07feb622-f41b-4070-ba7f-91dd5f74bbca', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'output_tokens': 28, 'input_tokens': 175, 'total_tokens'

In [8]:
# Несколько разных вопросов для демонстрации
questions = [
    "Какие требования нужны для мультиклассирования в паладина?",
    "Как работает вдохновение барда?",
    "Что происходит когда персонаж падает до 0 хитов?",
]

for question in questions:
    await ask_agent(question)
    print("\n" + "="*60 + "\n")

Вопрос: 'Какие требования нужны для мультиклассирования в паладина?'
[Агент вызывает инструмент]: qdrant-find
  Запрос: {'query': 'мультиклассирование паладин требования'}
[MCP вернул]: [{'type': 'text', 'text': '[\n  "Results for the query \'мультиклассирование паладин требования\'",\n  "<entry><content>умения Использование заклинаний; и \\nмультиклассирование</content><metadata></m...

[Ответ агента]:
Для мультиклассирования в паладина необходимо выполнить следующие требования:

1. **Минимальный показатель силы**: Ваш показатель силы должен быть не менее 13.
2. **Минимальный показатель харизмы**: Ваш показатель харизмы также должен быть не менее 13.

Эти требования указаны в разделе "Мультиклассирование" книги игрока (Player's Handbook), где подробно описаны условия для получения уровней в различных классах.



Вопрос: 'Как работает вдохновение барда?'
[Агент вызывает инструмент]: qdrant-find
  Запрос: {'query': 'bard inspiration'}
[MCP вернул]: [{'type': 'text', 'text': '[\n  "Resul